In [49]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
from sklearn.model_selection import train_test_split

In [50]:
#read data
sales = pd.read_csv("Dataset/Raw_data/Online_Sales.csv")
customers = pd.read_excel("Dataset/Raw_data/CustomersData.xlsx")
tax = pd.read_excel("Dataset/Raw_data/Tax_amount.xlsx")
discount = pd.read_csv("Dataset/Raw_data/Discount_Coupon.csv")
marketing = pd.read_csv("Dataset/Raw_data/Marketing_Spend.csv")

In [51]:
#data audit
datasets = {
    "sales": sales,
    "customers": customers,
    "tax": tax,
    "discount": discount,
    "marketing": marketing
}

for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print("="*40)
    print(df.shape)
    print(df.dtypes)
    print(df.isnull().sum())
    print(df.duplicated().sum())


SALES
(52924, 10)
CustomerID               int64
Transaction_ID           int64
Transaction_Date           str
Product_SKU                str
Product_Description        str
Product_Category           str
Quantity                 int64
Avg_Price              float64
Delivery_Charges       float64
Coupon_Status              str
dtype: object
CustomerID             0
Transaction_ID         0
Transaction_Date       0
Product_SKU            0
Product_Description    0
Product_Category       0
Quantity               0
Avg_Price              0
Delivery_Charges       0
Coupon_Status          0
dtype: int64
0

CUSTOMERS
(1468, 4)
CustomerID       int64
Gender             str
Location           str
Tenure_Months    int64
dtype: object
CustomerID       0
Gender           0
Location         0
Tenure_Months    0
dtype: int64
0

TAX
(20, 2)
Product_Category        str
GST                 float64
dtype: object
Product_Category    0
GST                 0
dtype: int64
0

DISCOUNT
(204, 4)
Month        

1. Sales

In [52]:
sales.head()

,CustomerID,Transaction_ID,Transaction_Date,Product_SKU,Product_Description,Product_Category,Quantity,Avg_Price,Delivery_Charges,Coupon_Status
0,17850,16679,1/1/2019,GGOENEBJ079499,Nest Learning Thermostat 3rd Gen-USA - Stainle...,Nest-USA,1,153.71,6.5,Used
1,17850,16680,1/1/2019,GGOENEBJ079499,Nest Learning Thermostat 3rd Gen-USA - Stainle...,Nest-USA,1,153.71,6.5,Used
2,17850,16681,1/1/2019,GGOEGFKQ020399,Google Laptop and Cell Phone Stickers,Office,1,2.05,6.5,Used
3,17850,16682,1/1/2019,GGOEGAAB010516,Google Men's 100% Cotton Short Sleeve Hero Tee...,Apparel,5,17.53,6.5,Not Used
4,17850,16682,1/1/2019,GGOEGBJL013999,Google Canvas Tote Natural/Navy,Bags,1,16.50,6.5,Used


In [53]:
sales["CustomerID"] = sales["CustomerID"].astype(str)

sales["Transaction_ID"] = sales["Transaction_ID"].astype(str)

sales["Transaction_Date"] = pd.to_datetime(
    sales["Transaction_Date"],
    errors="coerce"
)

In [54]:
text_cols = [
    "Product_SKU",
    "Product_Description",
    "Product_Category",
    "Coupon_Status"
]

for col in text_cols:
    sales[col] = (
        sales[col]
        .astype(str)
        .str.strip()
    )

In [55]:
numeric_cols = [
    "Quantity",
    "Avg_Price",
    "Delivery_Charges"
]

for col in numeric_cols:
    sales[col] = pd.to_numeric(
        sales[col],
        errors="coerce"
    )

In [56]:
#drop dul nhưng tránh drop nhầm order chứa nhiều items
sales = sales.drop_duplicates(
    subset=[
        "CustomerID",
        "Transaction_ID",
        "Product_SKU"
    ]
)

In [57]:
sales = sales[
    (sales["Quantity"] > 0) &
    (sales["Avg_Price"] > 0) &
    (sales["Delivery_Charges"] >= 0)
]

In [58]:
critical_cols = [
    "CustomerID",
    "Transaction_ID",
    "Transaction_Date",
    "Product_SKU"
]

sales = sales.dropna(
    subset=critical_cols
)

In [59]:
sales["Product_Description"] = sales[
    "Product_Description"
].fillna("Unknown")

sales["Coupon_Status"] = sales[
    "Coupon_Status"
].fillna("Unknown")

In [60]:
sales["Coupon_Status"] = (
    sales["Coupon_Status"]
    .str.title()
)
print(
    sales["Coupon_Status"]
    .value_counts()
)

Coupon_Status
Clicked     26926
Used        17904
Not Used     8094
Name: count, dtype: int64


In [61]:
# Tạo cột Month để join với Discount_Coupon
sales["Month"] = sales["Transaction_Date"].dt.strftime("%b")  # "Jan", "Feb"...

2. Customers

In [62]:
customers.head()

,CustomerID,Gender,Location,Tenure_Months
0,17850,M,Chicago,12
1,13047,M,California,43
2,12583,M,Chicago,33
3,13748,F,California,30
4,15100,M,California,49


In [63]:
customers["CustomerID"] = customers["CustomerID"].astype(str)

In [64]:
text_cols = ["Gender", "Location"]

for col in text_cols:
    customers[col] = (
        customers[col]
        .astype(str)
        .str.strip()
        .str.title()
    )

In [65]:
print(customers["Gender"].value_counts())

Gender
F    934
M    534
Name: count, dtype: int64


In [66]:
print(customers["Location"].value_counts())

Location
California       464
Chicago          456
New York         324
New Jersey       149
Washington Dc     75
Name: count, dtype: int64


In [67]:
customers["Tenure_Months"].describe()

count    1468.000000
mean       25.912125
std        13.959667
min         2.000000
25%        14.000000
50%        26.000000
75%        38.000000
max        50.000000
Name: Tenure_Months, dtype: float64

In [68]:
customers = customers[
    customers["Tenure_Months"] >= 0
]

3. Discount

In [69]:
discount.head()

,Month,Product_Category,Coupon_Code,Discount_pct
0,Jan,Apparel,SALE10,10
1,Feb,Apparel,SALE20,20
2,Mar,Apparel,SALE30,30
3,Jan,Nest-USA,ELEC10,10
4,Feb,Nest-USA,ELEC20,20


In [70]:
for col in ["Month", "Product_Category", "Coupon_Code"]:
    discount[col] = (
        discount[col]
        .astype(str)
        .str.strip()
    )

In [71]:
discount["Coupon_Code"] = (
    discount["Coupon_Code"]
    .str.upper()
)

In [72]:
print(discount["Month"].unique())

<ArrowStringArray>
['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov',
 'Dec']
Length: 12, dtype: str


In [73]:
discount = discount[
    (discount["Discount_pct"] >= 0) &
    (discount["Discount_pct"] <= 100)
]

In [74]:
#check category mismatch
discount_categories = set(
    discount["Product_Category"]
)

sales_categories = set(
    sales["Product_Category"]
)

print(
    discount_categories - sales_categories
)

{'Notebooks'}


In [75]:
print(sorted(sales["Product_Category"].unique()))
print(sorted(discount["Product_Category"].unique()))

['Accessories', 'Android', 'Apparel', 'Backpacks', 'Bags', 'Bottles', 'Drinkware', 'Fun', 'Gift Cards', 'Google', 'Headgear', 'Housewares', 'Lifestyle', 'More Bags', 'Nest', 'Nest-Canada', 'Nest-USA', 'Notebooks & Journals', 'Office', 'Waze']
['Accessories', 'Android', 'Apparel', 'Bags', 'Bottles', 'Drinkware', 'Gift Cards', 'Headgear', 'Housewares', 'Lifestyle', 'Nest', 'Nest-Canada', 'Nest-USA', 'Notebooks', 'Notebooks & Journals', 'Office', 'Waze']


4. Marketing

In [76]:
marketing.head()

,Date,Offline_Spend,Online_Spend
0,1/1/2019,4500,2424.50
1,1/2/2019,4500,3480.36
2,1/3/2019,4500,1576.38
3,1/4/2019,4500,2928.55
4,1/5/2019,4500,4055.30


In [77]:
marketing["Date"] = pd.to_datetime(
    marketing["Date"],
    errors="coerce"
)

In [78]:
marketing = marketing[
    (marketing["Offline_Spend"] >= 0) &
    (marketing["Online_Spend"] >= 0)
]

In [79]:
print(
    marketing["Date"]
    .isnull()
    .sum()
)

0


In [80]:
print(
    marketing["Date"]
    .duplicated()
    .sum()
)

0


5. Tax

In [81]:
tax.head()

,Product_Category,GST
0,Nest-USA,0.10
1,Office,0.10
2,Apparel,0.18
3,Bags,0.18
4,Drinkware,0.18


In [82]:
tax["Product_Category"] = (
    tax["Product_Category"]
    .astype(str)
    .str.strip()
)

In [83]:
tax = tax[
    (tax["GST"] >= 0) &
    (tax["GST"] <= 1)
]

In [84]:
#check có match với sales ko
tax_categories = set(
    tax["Product_Category"]
)

sales_categories = set(
    sales["Product_Category"]
)

print(
    tax_categories - sales_categories
)

set()


In [85]:
print(
    sales_categories - tax_categories
)

set()


6. Merge

In [86]:
# 1. Create month key
sales["Month"] = (
    sales["Transaction_Date"]
    .dt.strftime("%b")
)

# 2. Start from fact table
df = sales.merge(
    customers,
    on="CustomerID",
    how="left"
)

# 3. Tax
df = df.merge(
    tax,
    on="Product_Category",
    how="left"
)

# 4. Discount
df = df.merge(
    discount,
    on=["Product_Category", "Month"],
    how="left"
)

# 5. Marketing
df = df.merge(
    marketing,
    left_on="Transaction_Date",
    right_on="Date",
    how="left"
)

df.drop(columns=["Date"], inplace=True)

In [87]:
print("Sales:", sales.shape)
print("Final:", df.shape)

Sales: (52924, 11)
Final: (52924, 19)


In [88]:
df[
    ["Gender", "GST", "Discount_pct"]
].isnull().sum()

Gender            0
GST               0
Discount_pct    400
dtype: int64

In [89]:
df["Discount_pct"] = df[
    "Discount_pct"
].fillna(0)

df["Coupon_Code"] = df[
    "Coupon_Code"
].fillna("No_Coupon")

In [90]:
df[
    ["Gender", "GST", "Discount_pct"]
].isnull().sum()

Gender          0
GST             0
Discount_pct    0
dtype: int64

In [91]:
df.duplicated(
    subset=[
        "CustomerID",
        "Transaction_ID",
        "Product_SKU"
    ]
).sum()

np.int64(0)

7. Final cleaned files

In [94]:
df.head()

,CustomerID,Transaction_ID,Transaction_Date,Product_SKU,Product_Description,Product_Category,Quantity,Avg_Price,Delivery_Charges,Coupon_Status,Month,Gender,Location,Tenure_Months,GST,Coupon_Code,Discount_pct,Offline_Spend,Online_Spend
0,17850,16679,2019-01-01,GGOENEBJ079499,Nest Learning Thermostat 3rd Gen-USA - Stainle...,Nest-USA,1,153.71,6.5,Used,Jan,M,Chicago,12,0.10,ELEC10,10.0,4500,2424.5
1,17850,16680,2019-01-01,GGOENEBJ079499,Nest Learning Thermostat 3rd Gen-USA - Stainle...,Nest-USA,1,153.71,6.5,Used,Jan,M,Chicago,12,0.10,ELEC10,10.0,4500,2424.5
2,17850,16681,2019-01-01,GGOEGFKQ020399,Google Laptop and Cell Phone Stickers,Office,1,2.05,6.5,Used,Jan,M,Chicago,12,0.10,OFF10,10.0,4500,2424.5
3,17850,16682,2019-01-01,GGOEGAAB010516,Google Men's 100% Cotton Short Sleeve Hero Tee...,Apparel,5,17.53,6.5,Not Used,Jan,M,Chicago,12,0.18,SALE10,10.0,4500,2424.5
4,17850,16682,2019-01-01,GGOEGBJL013999,Google Canvas Tote Natural/Navy,Bags,1,16.50,6.5,Used,Jan,M,Chicago,12,0.18,AIO10,10.0,4500,2424.5


In [ ]:
#sales.to_csv("clean_sales.csv", index=False)
#customers.to_csv("clean_customers.csv", index=False)
#discount.to_csv("clean_discount.csv", index=False)
#marketing.to_csv("clean_marketing.csv", index=False)
#tax.to_csv("clean_tax.csv", index=False)

In [ ]:
#df.to_csv("customer_analytics_dataset.csv", index=False)